In [1]:

import json
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import GradientDescentOptimizer

symbols = ["H", "H", "H"]

def h3_ground_energy(bond_length):
    """
    Uses VQE to calculate the ground energy of the H3+ molecule with the given bond length.
    
    Args:
        - bond_length(float): The bond length of the H3+ molecule modelled as an
        equilateral triangle.
    Returns:
        - Union[float, np.tensor, np.array]: A float-like output containing the ground 
        state of the H3+ molecule with the given bond length.
    """
    geometry = np.array(
        [
            [0, 0, 0],
            [bond_length, 0, 0],
            [bond_length / 2, bond_length * np.sqrt(3) / 2, 0],
        ],
        requires_grad=False,
    )

    mol = qml.qchem.Molecule(symbols, geometry, charge=1)
    H, qubits = qml.qchem.molecular_hamiltonian(mol)

    singles, doubles = qml.qchem.excitations(mol.n_electrons, qubits)
    excitations = singles + doubles

    num_params = len(singles + doubles)
    params = np.zeros(num_params)

    hf_state = qml.qchem.hf_state(mol.n_electrons, qubits)

    dev = qml.device("default.qubit", wires=qubits)

    @qml.qnode(dev)
    def circuit(params, wires):
        qml.BasisState(hf_state, wires=wires)
        for i, excitation in enumerate(excitations):
            if len(excitation) == 4:
                qml.DoubleExcitation(params[i], wires=excitation)
            elif len(excitation) == 2:
                qml.SingleExcitation(params[i], wires=excitation)
        return qml.expval(H)

    optimizer = GradientDescentOptimizer(stepsize=0.4)

    # Perform VQE optimization
    max_iterations = 100
    conv_tol = 1e-06
    energy = circuit(params, wires=range(qubits))
    for n in range(max_iterations):
        params, prev_energy = optimizer.step_and_cost(circuit, params, wires=range(qubits))
        energy = circuit(params, wires=range(qubits))
        if np.abs(energy - prev_energy) <= conv_tol:
            break

    return energy

def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    outs = h3_ground_energy(ins)
    return str(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(
        solution_output, expected_output, atol=1e-4
    ), "Not the correct ground energy"


test_cases = [
    ("1.5", "-1.232574"),
    ("0.8", "-0.3770325"),
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")


Running test case 0 with input '1.5'...


Correct!
Running test case 1 with input '0.8'...


Correct!
